# CCM-109 — Deep Learning
## Aula 3 · Filtros de Convolução e Pooling

Neste notebook vamos:
1. Carregar uma imagem de teste
2. Aplicar **filtros predefinidos** (detecção de borda, nitidez, desfoque, relevo) e visualizar os *feature maps* resultantes
3. Explorar as operações de **Max-Pooling** e **Average Pooling** e comparar seus efeitos

Usamos `torch.nn.functional` para as operações de forma explícita — os pesos dos filtros são definidos por nós, não aprendidos.

## 0. Dependências

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from skimage import data, color

plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.titlesize'] = 9
plt.rcParams['axes.titlepad'] = 4

print(f'PyTorch {torch.__version__}')

## 1. Carregando a imagem

Usamos `skimage.data.camera()` — imagem grayscale clássica de 512×512 pixels, sempre disponível sem download.

In [ ]:
# Imagem grayscale para filtros (512×512)
img_gray_np = data.camera().astype(np.float32) / 255.0  # normaliza para [0, 1]

# Imagem colorida para pooling (astronaut, 512×512×3)
img_color_np = data.astronaut().astype(np.float32) / 255.0

# Converte para tensores PyTorch no formato (batch, canais, H, W)
img_gray = torch.tensor(img_gray_np).unsqueeze(0).unsqueeze(0)   # (1, 1, 512, 512)
img_color = torch.tensor(img_color_np).permute(2, 0, 1).unsqueeze(0)  # (1, 3, 512, 512)

print(f'Grayscale: {img_gray.shape}   dtype={img_gray.dtype}')
print(f'Color    : {img_color.shape}  dtype={img_color.dtype}')

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].imshow(img_gray_np, cmap='gray'); axes[0].set_title('Camera (grayscale)'); axes[0].axis('off')
axes[1].imshow(img_color_np);             axes[1].set_title('Astronaut (RGB)');    axes[1].axis('off')
plt.tight_layout()
plt.show()

## 2. Filtros predefinidos

Definimos manualmente os pesos dos filtros 3×3. Em uma CNN treinada, esses pesos seriam **aprendidos** via gradiente descendente; aqui os escolhemos deliberadamente para entender o efeito de cada um.

| Filtro | Efeito |
|---|---|
| Blur (box) | Suaviza, remove ruído |
| Blur (Gaussiano) | Suaviza preservando mais estrutura |
| Sobel H | Detecta bordas horizontais |
| Sobel V | Detecta bordas verticais |
| Laplaciano | Detecta bordas em todas as direções |
| Nitidez | Realça detalhes |
| Relevo (*emboss*) | Efeito 3-D de relevo |

In [ ]:
filters = {
    'Original': None,  # placeholder

    'Blur\n(box)': np.array([
        [1, 1, 1],
        [1, 1, 1],
        [1, 1, 1],
    ], dtype=np.float32) / 9.0,

    'Blur\n(Gaussiano)': np.array([
        [1, 2, 1],
        [2, 4, 2],
        [1, 2, 1],
    ], dtype=np.float32) / 16.0,

    'Sobel H\n(bordas horiz.)': np.array([
        [-1, -2, -1],
        [ 0,  0,  0],
        [ 1,  2,  1],
    ], dtype=np.float32),

    'Sobel V\n(bordas vert.)': np.array([
        [-1, 0, 1],
        [-2, 0, 2],
        [-1, 0, 1],
    ], dtype=np.float32),

    'Laplaciano\n(todas bordas)': np.array([
        [ 0, -1,  0],
        [-1,  4, -1],
        [ 0, -1,  0],
    ], dtype=np.float32),

    'Nitidez\n(sharpening)': np.array([
        [ 0, -1,  0],
        [-1,  5, -1],
        [ 0, -1,  0],
    ], dtype=np.float32),

    'Relevo\n(emboss)': np.array([
        [-2, -1,  0],
        [-1,  1,  1],
        [ 0,  1,  2],
    ], dtype=np.float32),
}

# Visualiza os kernels (exceto o placeholder)
named = [(n, k) for n, k in filters.items() if k is not None]
fig, axes = plt.subplots(2, 4, figsize=(11, 5))
axes = axes.flatten()
for ax, (name, kernel) in zip(axes, named):
    im = ax.imshow(kernel, cmap='RdBu_r', vmin=-max(abs(kernel.min()), abs(kernel.max())),
                           vmax= max(abs(kernel.min()), abs(kernel.max())))
    ax.set_title(name, fontsize=8)
    ax.set_xticks(range(3)); ax.set_yticks(range(3))
    for r in range(3):
        for c in range(3):
            ax.text(c, r, f'{kernel[r,c]:.2f}', ha='center', va='center',
                    fontsize=7, color='black' if abs(kernel[r,c]) < 0.5 else 'white')
axes[-1].axis('off')
plt.suptitle('Kernels dos filtros (azul = negativo, vermelho = positivo)', fontsize=9, y=1.01)
plt.tight_layout()
plt.show()

## 3. Aplicando os filtros — Feature Maps

Usamos `F.conv2d` com `padding=1` para manter o tamanho de saída igual ao de entrada (same-padding).

**Como funciona `F.conv2d`:**
```
F.conv2d(input, weight, bias=None, stride=1, padding=0)
  input  : (batch, C_in,  H,   W)
  weight : (C_out, C_in, kH, kW)   ← nosso kernel tem C_in=1, C_out=1
  output : (batch, C_out, H', W')
```

In [ ]:
def apply_filter(img_tensor, kernel_np):
    """Aplica kernel 3×3 em imagem grayscale (1,1,H,W) com same-padding."""
    weight = torch.tensor(kernel_np).unsqueeze(0).unsqueeze(0)  # (1,1,3,3)
    with torch.no_grad():
        out = F.conv2d(img_tensor, weight, padding=1)
    return out.squeeze().numpy()  # (H, W)


def normalize_for_display(arr):
    """Normaliza para [0,1] para exibição, tratando filtros com valores negativos."""
    arr = arr - arr.min()
    if arr.max() > 0:
        arr = arr / arr.max()
    return arr


# Aplica todos os filtros
results = {}
for name, kernel in filters.items():
    if kernel is None:
        results[name] = img_gray_np
    else:
        results[name] = normalize_for_display(apply_filter(img_gray, kernel))

# Plota
n = len(results)
cols = 4
rows = (n + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(13, 6.5))
axes = axes.flatten()

for ax, (name, feat_map) in zip(axes, results.items()):
    ax.imshow(feat_map, cmap='gray')
    ax.set_title(name, fontsize=8)
    ax.axis('off')

for ax in axes[n:]:
    ax.axis('off')

plt.suptitle('Feature maps — efeito de cada filtro', fontsize=10, y=1.01)
plt.tight_layout()
plt.show()

### 3.1 Zoom — comparando detalhe dos filtros de borda

Sobel H, Sobel V e Laplaciano de perto numa região com bordas marcadas (rosto do fotografo).

In [ ]:
# Recorte central da imagem (rosto + câmera)
r0, r1, c0, c1 = 50, 280, 100, 380

edge_filters = ['Original', 'Sobel H\n(bordas horiz.)', 'Sobel V\n(bordas vert.)', 'Laplaciano\n(todas bordas)']

fig, axes = plt.subplots(1, 4, figsize=(13, 3.5))
for ax, name in zip(axes, edge_filters):
    crop = results[name][r0:r1, c0:c1]
    ax.imshow(crop, cmap='gray')
    ax.set_title(name.replace('\n', ' '), fontsize=8)
    ax.axis('off')

plt.suptitle('Zoom — filtros de detecção de borda', fontsize=9)
plt.tight_layout()
plt.show()

### 3.2 Combinando Sobel H e Sobel V — magnitude do gradiente

Na prática detectores de borda calculam a **magnitude** $\|G\| = \sqrt{G_H^2 + G_V^2}$, combinando as duas direções.

In [ ]:
sobel_h_raw = apply_filter(img_gray, filters['Sobel H\n(bordas horiz.)'])
sobel_v_raw = apply_filter(img_gray, filters['Sobel V\n(bordas vert.)'])
magnitude   = np.sqrt(sobel_h_raw**2 + sobel_v_raw**2)
magnitude   = normalize_for_display(magnitude)

fig, axes = plt.subplots(1, 3, figsize=(11, 3.5))
titles = ['Sobel H (bordas horiz.)', 'Sobel V (bordas vert.)', 'Magnitude combinada  √(H²+V²)']
maps   = [normalize_for_display(sobel_h_raw), normalize_for_display(sobel_v_raw), magnitude]

for ax, title, m in zip(axes, titles, maps):
    ax.imshow(m, cmap='gray')
    ax.set_title(title, fontsize=8)
    ax.axis('off')

plt.tight_layout()
plt.show()

## 4. Max-Pooling vs Average Pooling

Pooling **reduz a resolução espacial** (sub-amostragem), criando invariância a pequenas translações.

| | Max-Pooling | Average Pooling |
|---|---|---|
| **Operação** | máximo da janela | média da janela |
| **Preserva** | características salientes / bordas | textura geral / gradientes suaves |
| **Uso típico** | camadas intermediárias de CNNs | camada final (*global average pooling*) |

Vamos testar com a imagem colorida **astronaut** e kernels 2×2 e 4×4 para ver o efeito da escala.

In [ ]:
def to_numpy_img(tensor):
    """(1, C, H, W) → (H, W, C) float32 clampado em [0,1]"""
    return tensor.squeeze(0).permute(1, 2, 0).numpy().clip(0, 1)

kernel_sizes = [2, 4, 8]

pooling_results = {'original': to_numpy_img(img_color)}
for k in kernel_sizes:
    pooling_results[f'max_{k}x{k}'] = to_numpy_img(F.max_pool2d(img_color, kernel_size=k, stride=k))
    pooling_results[f'avg_{k}x{k}'] = to_numpy_img(F.avg_pool2d(img_color, kernel_size=k, stride=k))

print('Resolução após cada operação:')
print(f'  Original  : {pooling_results["original"].shape[:2]}')
for k in kernel_sizes:
    h, w = pooling_results[f'max_{k}x{k}'].shape[:2]
    print(f'  Pool {k}×{k}  : {h}×{w}  (↓{k}×)')

In [ ]:
fig = plt.figure(figsize=(14, 7))
gs  = gridspec.GridSpec(2, 4, figure=fig, hspace=0.35, wspace=0.05)

# Imagem original ocupa as 2 primeiras linhas na coluna 0
ax_orig = fig.add_subplot(gs[:, 0])
ax_orig.imshow(pooling_results['original'])
ax_orig.set_title(f'Original\n512×512', fontsize=8)
ax_orig.axis('off')

for col, k in enumerate(kernel_sizes, start=1):
    h, w = pooling_results[f'max_{k}x{k}'].shape[:2]

    ax_max = fig.add_subplot(gs[0, col])
    ax_max.imshow(pooling_results[f'max_{k}x{k}'])
    ax_max.set_title(f'Max-Pool {k}×{k}\n{h}×{w}', fontsize=8)
    ax_max.axis('off')

    ax_avg = fig.add_subplot(gs[1, col])
    ax_avg.imshow(pooling_results[f'avg_{k}x{k}'])
    ax_avg.set_title(f'Avg-Pool {k}×{k}\n{h}×{w}', fontsize=8)
    ax_avg.axis('off')

# Rótulos de linha
fig.text(0.28, 0.97, 'MAX POOLING',     ha='center', fontsize=9, fontweight='bold', color='#2563eb')
fig.text(0.28, 0.50, 'AVERAGE POOLING', ha='center', fontsize=9, fontweight='bold', color='#059669')

plt.suptitle('Max-Pooling vs Average Pooling — efeito da redução de resolução', fontsize=10, y=1.02)
plt.show()

### 4.1 Diferença pixel a pixel: Max − Average

A diferença mostra onde cada operação diverge: Max-Pooling preserva picos de intensidade (bordas, brilhos), enquanto Average Pooling os suaviza.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))

for ax, k in zip(axes, kernel_sizes):
    diff = pooling_results[f'max_{k}x{k}'].mean(axis=2) - pooling_results[f'avg_{k}x{k}'].mean(axis=2)
    vmax = np.abs(diff).max()
    im = ax.imshow(diff, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
    ax.set_title(f'Max − Avg  (pool {k}×{k})', fontsize=8)
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02)

plt.suptitle('Diferença (Max − Average) por canal de luminância\n'
             'Vermelho = Max > Avg (bordas/brilhos preservados)  |  Azul = Avg > Max', fontsize=8)
plt.tight_layout()
plt.show()

### 4.2 Pooling sobre feature map de borda

Uma forma intuitiva de ver a diferença: aplique o filtro Laplaciano (detecção de bordas) e depois aplique cada tipo de pooling. Max-Pooling **mantém** as respostas de borda mais fortes; Average Pooling as **dilui**.

In [ ]:
# Feature map de bordas (Laplaciano)
lap_raw    = apply_filter(img_gray, filters['Laplaciano\n(todas bordas)'])
lap_tensor = torch.tensor(lap_raw).unsqueeze(0).unsqueeze(0)  # (1,1,H,W)

lap_max2 = F.max_pool2d(lap_tensor, 2, 2).squeeze().numpy()
lap_avg2 = F.avg_pool2d(lap_tensor, 2, 2).squeeze().numpy()
lap_max4 = F.max_pool2d(lap_tensor, 4, 4).squeeze().numpy()
lap_avg4 = F.avg_pool2d(lap_tensor, 4, 4).squeeze().numpy()

items = [
    ('Feature map original\n(Laplaciano 512×512)', lap_raw),
    ('Max-Pool 2×2\n(256×256)',                    lap_max2),
    ('Avg-Pool 2×2\n(256×256)',                    lap_avg2),
    ('Max-Pool 4×4\n(128×128)',                    lap_max4),
    ('Avg-Pool 4×4\n(128×128)',                    lap_avg4),
]

fig, axes = plt.subplots(1, 5, figsize=(15, 3.2))
for ax, (title, arr) in zip(axes, items):
    ax.imshow(normalize_for_display(arr), cmap='inferno')
    ax.set_title(title, fontsize=7.5)
    ax.axis('off')

plt.suptitle('Pooling sobre feature map de bordas — Max preserva picos, Avg os dilui', fontsize=9)
plt.tight_layout()
plt.show()

## 5. Global Average Pooling (GAP)

**Global Average Pooling** colapsa todo o feature map de tamanho H×W para um único valor por canal — equivalente a `F.adaptive_avg_pool2d(x, (1,1))`. É usado no final de arquiteturas modernas (ResNet, MobileNet) em substituição ao `Flatten + Dense`.

**Por que isso funciona?** Cada canal do feature map representa um conceito aprendido (ex: "orelha de gato", "roda de carro"). O GAP computa a ativação média desse conceito na imagem inteira, produzindo um vetor de features muito compacto e resistente a *overfitting*.

In [ ]:
# Simula uma saída de bloco convolucional: 8 canais (feature maps artificiais)
torch.manual_seed(42)
fake_feature_map = torch.rand(1, 8, 16, 16)  # (batch=1, canais=8, H=16, W=16)

# GAP → (1, 8, 1, 1) → (8,)
gap_output = F.adaptive_avg_pool2d(fake_feature_map, (1, 1)).squeeze()

fig, axes = plt.subplots(2, 5, figsize=(13, 4.5))

# Mostra os 8 feature maps
for i in range(8):
    ax = axes[i // 4][i % 4]
    ax.imshow(fake_feature_map[0, i].numpy(), cmap='viridis', vmin=0, vmax=1)
    ax.set_title(f'Canal {i}  (média={gap_output[i]:.2f})', fontsize=7.5)
    ax.axis('off')

# Mostra o vetor GAP
for idx in (4, 9):
    axes.flatten()[idx].axis('off')

ax_bar = fig.add_subplot(1, 5, 5)
ax_bar.barh(range(8), gap_output.numpy(), color=plt.cm.viridis(gap_output.numpy()))
ax_bar.set_yticks(range(8))
ax_bar.set_yticklabels([f'Canal {i}' for i in range(8)], fontsize=7)
ax_bar.set_xlabel('Ativação média', fontsize=7)
ax_bar.set_title('Vetor GAP (8-d)', fontsize=8)
ax_bar.tick_params(labelsize=7)

plt.suptitle('Global Average Pooling — cada feature map 16×16 vira um único número', fontsize=9)
plt.tight_layout()
plt.show()

print(f'\nEntrada: {tuple(fake_feature_map.shape)}  →  GAP output: {tuple(gap_output.shape)}')

## 6. Resumo

| Conceito | O que faz | Por que importa |
|---|---|---|
| **Convolução** | Aplica um kernel deslizante sobre a entrada | Detecta padrões locais (bordas, texturas…) |
| **Feature map** | Saída após aplicar o filtro | Cada mapa destaca um tipo de padrão |
| **Max-Pool** | Mantém o máximo de cada janela | Preserva detalhes / bordas, reduz resolução |
| **Avg-Pool** | Calcula a média de cada janela | Suaviza, cria invariância translacional |
| **GAP** | Avg-Pool global → 1 valor por canal | Substitui Flatten, menos parâmetros |

---

**Próximo passo:** nos notebooks `lec03-keras.ipynb` e `lec03-torch.ipynb` esses blocos convolucionais são **treinados** em dados reais usando transfer learning a partir de uma ResNet pré-treinada no ImageNet.